# Kateto State-Tuning (Capa 2: Voz Seco Multi-Turno) — Google Colab (GPU T4)

Este notebook entrena **únicamente el vector de estado inicial S_0** (100% de los pesos del modelo congelados).
- **Tiempo estimado:** ~6 a 8 minutos en GPU T4 gratuita de Colab.
- **Salida:** `rwkv-4.pth` (21 MB) que se descarga automáticamente a tu PC.
- **Modelo Base:** `rwkv7-g1j-2.9b-20260831-ctx16384.pth` (BlinkDL oficial, limpio, sin contaminación de noticias).
- **Dataset:** `rwkv_kateto_seco.jsonl` (480 ejemplos, 79.2% diálogos multi-turno en Rioplatense, 1ra persona).

In [ ]:
# 1) Dependencias y verificación de GPU
!pip uninstall -y torchao
!pip install -q --no-cache-dir rwkv rwkv-fla einops jsonlines peft datasets triton huggingface_hub "lightning>=2.0.0"
!nvidia-smi
import torch
assert torch.cuda.is_available(), "ERROR: Activa la GPU en Entorno de ejecución > Cambiar tipo de entorno > T4 GPU"

In [ ]:
# 2) Descargar el modelo base oficial RWKV-7 2.9B (~5.5 GB)
import os
from huggingface_hub import hf_hub_download

os.makedirs('/content/models', exist_ok=True)
model_path = hf_hub_download(
    repo_id='BlinkDL/rwkv7-g1',
    filename='rwkv7-g1j-2.9b-20260831-ctx16384.pth',
    local_dir='/content/models'
)
print("Modelo descargado en:", model_path)

In [ ]:
# 3) Clonar RWKV-PEFT con el fix de Loss Masking para ChatML
!git clone https://github.com/Joluck/RWKV-PEFT /content/RWKV-PEFT

# Aplicar el fix de loss_mask en dataset.py para que <|im_user|> esté enmascarado
fix_code = '''
with open('/content/RWKV-PEFT/rwkvt/dataset/dataset.py', 'r') as f:
    code = f.read()

if '<|im_user|>' not in code:
    old_block = '''        mask_fn = mask_fn_dict.get(args.loss_mask)

        if mask_fn!=None:
            t1 = pipeline.encode('User:')
            t2 = pipeline.encode('Assistant:')
            y = mask_fn(dix, t1, t2, min_len)'''
    new_block = '''        mask_fn = mask_fn_dict.get(args.loss_mask)

        if mask_fn!=None:
            if "<|im_user|>" in ctx:
                t1 = pipeline.encode('<|im_user|>')
                t2 = pipeline.encode('<|im_start|>')
            else:
                t1 = pipeline.encode('User:')
                t2 = pipeline.encode('Assistant:')
            y = mask_fn(dix, t1, t2, min_len)'''
    code = code.replace(old_block, new_block)
    with open('/content/RWKV-PEFT/rwkvt/dataset/dataset.py', 'w') as f:
        f.write(code)
    print("✓ Fix de Loss Mask ChatML aplicado exitosamente en dataset.py")
else:
    print("El fix ya estaba presente.")
'''
exec(fix_code)

In [ ]:
# 4) Subir el archivo rwkv_kateto_seco.jsonl (o descargarlo)
from google.colab import files
import os

print("Subí el archivo 'rwkv_kateto_seco.jsonl' generado en tu PC:")
uploaded = files.upload()
fname = list(uploaded.keys())[0]
os.makedirs('/content/data', exist_ok=True)
os.replace(fname, '/content/data/rwkv_kateto_seco.jsonl')
print(f"✓ Dataset cargado: /content/data/rwkv_kateto_seco.jsonl ({os.path.getsize('/content/data/rwkv_kateto_seco.jsonl')} bytes)")

In [ ]:
# 5) Ejecutar State Tuning (5 épocas, ~6 minutos)
import subprocess

cmd = [
    "python", "train.py",
    "--load_model", "/content/models/rwkv7-g1j-2.9b-20260831-ctx16384.pth",
    "--proj_dir", "/content/out_state",
    "--data_file", "/content/data/rwkv_kateto_seco.jsonl",
    "--data_type", "jsonl",
    "--loss_mask", "qa",
    "--vocab_size", "65536",
    "--n_layer", "32",
    "--n_embd", "2560",
    "--ctx_len", "512",
    "--micro_bsz", "1",
    "--epoch_steps", "500",
    "--epoch_count", "5",
    "--epoch_save", "1",
    "--lr_init", "1e-2",
    "--lr_final", "1e-4",
    "--accelerator", "gpu",
    "--precision", "bf16",
    "--devices", "1",
    "--strategy", "auto",
    "--grad_cp", "1",
    "--my_testing", "x070",
    "--op", "fla",
    "--peft", "state"
]

print(">>> INICIANDO STATE-TUNING DE KATETO SECO <<<")
res = subprocess.run(cmd, cwd="/content/RWKV-PEFT")
assert res.returncode == 0, f"Entrenamiento falló con código {res.returncode}"
print(">>> STATE-TUNING COMPLETADO EXITOSAMENTE <<<")

In [ ]:
# 6) Descargar el checkpoint de estado final (rwkv-4.pth, 21 MB)
import glob
from google.colab import files

states = sorted(glob.glob('/content/out_state/*.pth'))
print("Estados generados:", states)
final_state = states[-1]
print(f"Descargando {final_state} a tu computadora...")
files.download(final_state)
print("Guardalo en tu carpeta del proyecto en: out/rwkv_states/seco/rwkv-4.pth")